#### Part 1: Exploration and Data Quality

In [8]:
# Read the raw data files and summarize them
from pathlib import Path
import pandas as pd

RAW = Path("../data/raw")

FILES = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "geo": "olist_geolocation_dataset.csv",
    "cat_en": "product_category_name_translation.csv",
}

d = {k: pd.read_csv(RAW / v) for k, v in FILES.items()}

summary = pd.DataFrame(
    [
        {
            "table": k,
            "filename": FILES[k],
            "rows": v.shape[0],
            "n_cols": v.shape[1],
            "columns": ", ".join(v.columns),
        }
        for k, v in d.items()
    ]
).set_index("table")

pd.set_option("display.max_colwidth", None)
summary

,filename,rows,n_cols,columns
table,,,,
orders,olist_orders_dataset.csv,99441,8,"order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date"
items,olist_order_items_dataset.csv,112650,7,"order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value"
payments,olist_order_payments_dataset.csv,103886,5,"order_id, payment_sequential, payment_type, payment_installments, payment_value"
reviews,olist_order_reviews_dataset.csv,99224,7,"review_id, order_id, review_score, review_comment_title, review_comment_message, review_creation_date, review_answer_timestamp"
customers,olist_customers_dataset.csv,99441,5,"customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state"
sellers,olist_sellers_dataset.csv,3095,4,"seller_id, seller_zip_code_prefix, seller_city, seller_state"
products,olist_products_dataset.csv,32951,9,"product_id, product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm"
geo,olist_geolocation_dataset.csv,1000163,5,"geolocation_zip_code_prefix, geolocation_lat, geolocation_lng, geolocation_city, geolocation_state"
cat_en,product_category_name_translation.csv,71,2,"product_category_name, product_category_name_english"


##### The raw dataset consists of 9 tables:
1. Orders (99,441 rows × 8 columns): Contains order-level information, including order ID, customer ID, order status, and order timestamps.
2. Items (112,650 rows × 7 columns): Contains details of items in each order, including order ID, product ID, seller ID, shipping limit date, price, and freight value.
3. Payments (103,886 rows × 5 columns): Contains payment details for each order, including payment type, payment installments, payment sequence, and payment value.
4. Reviews (99,224 rows × 7 columns): Contains customer feedback for each order, including review score, review comments, and review timestamps.
5. Customers (99,441 rows × 5 columns): Contains customer identifiers and location details, including unique customer ID, ZIP code prefix, city, and state.
6. Sellers (3,095 rows × 4 columns): Contains seller identifiers and location details, including ZIP code prefix, city, and state.
7. Products (32,951 rows × 9 columns): Contains product attributes, including category, name and description lengths, number of photos, weight, and dimensions.
8. Geolocation (geo) (1,000,163 rows × 5 columns): Contains ZIP code prefixes and geographical information, including latitude, longitude, city, and state.
9. Category Translation (cat_en) (71 rows × 2 columns): Maps Portuguese product category names to their English equivalents for analysis.

In [ ]:
# This function is used to profile a dataframe and provide information about its shape
# Uniqueness of a specified key column, number of duplicated rows, data types, missing values, and unique values for each column.
def profile(df, key=None):
    print(f"shape: {df.shape}")
    if key:
        print(f"key {key!r} unique: {df[key].is_unique} "
              f"({df[key].nunique()} distinct of {len(df)} rows)")
    print(f"fully duplicated rows: {df.duplicated().sum()}")
    out = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(),
    })
    return out

In [12]:
orders = d["orders"].copy()
profile(orders, key="order_id")

shape: (99441, 8)
key 'order_id' unique: True (99441 distinct of 99441 rows)
fully duplicated rows: 0


,dtype,n_missing,pct_missing,n_unique
order_id,str,0,0.00,99441
customer_id,str,0,0.00,99441
order_status,str,0,0.00,8
order_purchase_timestamp,str,0,0.00,98875
order_approved_at,str,160,0.16,90733
order_delivered_carrier_date,str,1783,1.79,81018
order_delivered_customer_date,str,2965,2.98,95664
order_estimated_delivery_date,str,0,0.00,459


##### Oberservations of "orders table"
- order_id is unique across all 99,441 rows, and there are no fully duplicated rows. So one row per order is a valid grain for the base table.
- Dtypes: pandas shows str. It means the same thing here: nothing is parsed as a date yet.
- Missing values. Approved has 160 missing, carrier handoff 1,783, and customer delivery 2,965. These follow the order lifecycle (purchase, then approval, then carrier handoff, then delivery), so a missing later stage usually means an earlier stage never completed, not that data was lost. That's the "event never happened" kind of missing the brief wants us to separate from collection gaps.
- order_purchase_timestamp has 98,875 distinct values for 99,441 rows. That's just several orders placed in the same second. It isn't a duplicate problem, because order_id is unique.
- order_estimated_delivery_date has only 459 distinct values. It's a date with no time of day, so we'll compare it to delivery dates at day level.
The 160 orders with no approval time matter for Part 2. Our prediction point is order approval, so an order that was never approved has no prediction moment. We'll decide what to do with them later.


In [ ]:
# Count of each order status
orders["order_status"].value_counts(dropna=False)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

What the status counts show

Delivered is 96,478 (97.0%). The other 2,963 orders never reached the customer according to the data.
The statuses are stages of the lifecycle: created → approved → invoiced → processing → shipped → delivered. Then there are two exits, canceled and unavailable.
shipped is the biggest non-delivered group (1,107). Those left the warehouse but have no delivery record. Some may be genuinely lost or late, and some may be recent orders that were still in transit when the data was extracted. That is a censoring question for the time split in Part 2.

A mismatch to explain. Non-delivered orders total 2,963, but 2,965 orders have no customer delivery date. If missing delivery date meant "not delivered", the numbers would match exactly. They differ by 2, so a few orders have a status that disagrees with their timestamps.